In [ ]:
# import libraries
import pandas as pd
import numpy as np
import os
import re
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# VAR for the country data.
# extract country data.
target_file_name = "Country Data (1979Q2-2023Q3).xls"
full_path_folder = os.getcwd() + "\\" + target_file_name


country_df = pd.read_excel(full_path_folder, sheet_name = None)
country_df

In [ ]:
target_df = country_df['JAPAN']
target_df

#quarter_string = "1979Q2"

def quarter_date_converter(target_date_str):
    # Convert date string into datetime. 
    regex_str = r"(\d{4})Q([1-4])"
    match = re.match(regex_str, target_date_str)
    
    # Q1 = Mar 2026/ month  = 3
    # Q2 = Jun 2026/ month = 6
    # Q3 = Sep 2026/ month = 9
    # Q4 = Dec 2026/ month = 12
    
    if match:
        # extract the captured groups
        year = int(match.group(1)) # YYYY
        #print(year)
        quarter = int(match.group(2)) # Q#
        month = (quarter) * 3 # 1 x 3 = 3 march, 2 x 3 = 6 jun
        # create clean datetime object. 
        clean_date = pd.Timestamp(year = year, month = month, day = 1) + pd.offsets.QuarterEnd(0)
    
    return clean_date

In [ ]:
target_df['new_date'] = target_df['date'].apply(quarter_date_converter)
target_df = target_df.drop(columns = ['date'])
new_cols_list = [col for col in target_df.columns.tolist() if col != 'new_date']
new_cols_list = ['new_date'] + new_cols_list
target_df = target_df[new_cols_list]
target_df

# put the date into index. 
target_clean_df = target_df.copy()
target_clean_df.set_index('new_date', inplace = True)
target_clean_df.sort_index(inplace = True)
display(target_clean_df)

In [ ]:
# lets implement the VAR. 
# First of all, lets do a proper testing for the stationarity of the variables in JAPAN. 
# if there are any variables that are non-stationary we will have to difference them.

def run_adf_test(series, variable_name):
#    print(f"=== ADF Test Results for Variable: {variable_name} ===")

    # we use maxlag=None to let AIC automatically choose the optimal test lags
    result = adfuller(series.dropna(), autolag = 'AIC')

#    print(f"ADF Statistic: {result[0]:.6f}")
#    print(f"p-value:       {result[1]:.6f}")
#    print(f"Lags used:     {result[2]}")
#    print(f"Observations:  {result[3]}")
#    print(f"Critical Values:")
#    for key,value in result[4].items():
#        print(f"    {key}: {value:.6f}")
        
#    print(f"result[4]['5%'] is: \n {result[4]['5%']}")
    
    # 3. Automated Interpretation Matrix
#    if result[1] <= 0.05:
#        print("\nVerdict: STATIONARY. The p-value is <= 0.05.")
#        print("You can safely use this variable in your VAR model as-is.\n")
#    else:
#        print("\nVerdict: NON-STATIONARY. The p-value is > 0.05.")
#        print("This variable has a trend. You must take its first difference (diff()) before running the VAR.\n")

    # i want to see these in a table. 
    # lets return the adf stats, p-value, lags used, crit_vals [5%], 
    # create a dictionary.
    adf_stats_dict = {
        'target_var' : variable_name,
        'adf_stats' : result[0],
        'p_value' : result[1],
        'lags_used' : result[2],
        'nobs' : result[3],
        'crit_val_5' : result[4]['5%']        
    }

    medium_df = pd.DataFrame.from_dict(adf_stats_dict, orient = 'index').T 

    # Create a column to see if its stationary or not. 
    medium_df['stationary'] = np.where(medium_df['p_value'] <= 0.05, "Y", "N")
    
    return medium_df

def check_adf_stats(input_df):
    output_df = None
    for col in input_df.columns.tolist():
        #print(col)
        medium_df2 = run_adf_test(input_df[col], col)
        if output_df is None:
#            print(f"output_df is not defined and we are here.")
            output_df = medium_df2
        else:
#            print(f"output_df exist and we are now appending it.")
            output_df = pd.concat([output_df, medium_df2], axis = 0)
    return output_df
#    display(adf_stats_df)

adf_stats_df = check_adf_stats(target_clean_df)
adf_stats_df

In [ ]:
target_cols = adf_stats_df['target_var'].loc[adf_stats_df['stationary'] == "N"].values.tolist()
non_target_cols = [col for col in target_clean_df.columns.tolist() if col not in target_cols]
print(non_target_cols)
target_clean_df2 = target_clean_df[target_cols].diff()
#display(target_clean_df2)
non_target_clean_df = target_clean_df[non_target_cols]
#display(non_target_clean_df)
comb_clean_df = non_target_clean_df.merge(target_clean_df2, left_index = True, right_index = True, how = 'left').dropna()
display(comb_clean_df)

adf_stats_df = check_adf_stats(comb_clean_df)
display(adf_stats_df)

In [ ]:
model = VAR(comb_clean_df)
# let statsmodels automatically find the best lag length using AIC

lag_results = model.select_order(maxlags=5)
# extract the absolute optimal lag chosen specifically for AIC. 
optimal_aic_lag = lag_results.aic
print(f"\n--- Best Lag Selection ---")
print(lag_results.summary())
print(f"optimal_aic_lag is: \n {optimal_aic_lag}")

lagged_comb_df = comb_clean_df.copy()
lagged_comb_df = lagged_comb_df.shift(1).add_suffix('_lag1')

# concatenate the data. 
total_df = pd.concat([comb_clean_df, lagged_comb_df], axis = 1)
correl_matrix = total_df.dropna().corr()
#display(correl_matrix)

# export to excel. 
correl_matrix.to_excel('correl_matrix_japan.xlsx', sheet_name = 'correl_matrx', index = True)
print(f"correl_matrix_japan.xlsx has been exported to ({os.getcwd}\\correl_matrix_japan.xlsx)")

In [ ]:
# create an interactive correlation matrix heatmap via plotly. 

# 1. Create the interactive heatmap
fig = px.imshow(
    correl_matrix,
    text_auto = '.4f', # i want it in 4 decimals.
    color_continuous_scale = [
        [0.0, '#FF433D'], # Intense Terminal Red for deep negative correlations
        [0.5, '#1A1A1A'], # Dark Muted Charcoal baseline for absolute zero neutrality
        [1.0, '#0068ff'], # Intense Terminal Cyan/Blue for deep positive correlation
    ],
    zmin = -1, zmax = 1, # locks the color bar to the standard correlation limits.
    labels = dict(color = "Correlation"), # Custom legend title.
    title = "Interactive Correlation Matrix Heatmap (Japan Macro Vars)",
    # Tells plotly to fill the wide canvas instead of keeping squares.
    aspect = "auto",
)

# 1.5. Add this line to increase the number font size
fig.update_traces(textfont_size = 12,
                 textfont_color = "#FFFFFF",
                # Added: Creates 1-pixel gridlines between all boxes
                xgap = 1,
                ygap = 1
                 ) 

# 2. Polish the layout 
fig.update_layout(
#    title_font_size = 18, 
#    title_x = 0.5, # Centers the title
    width = 1800, # set equal dimensions perfectly square cells
    height = 800,

    # Background canvas architecture.
    paper_bgcolor = '#000000', # Surrounding interface panel background
    plot_bgcolor = '#000000',  # active chart plotting grid matrix background

    # typography elements
    title_font_size = 20,
    title_font_color = '#FFA028', # Classic Bloomberg Amber header font
    title_x = 0.5,

    # Axist tick control config
    
    xaxis_nticks = len(correl_matrix.columns),
    yaxis_nticks = len(correl_matrix.columns),

    # legend and scale panel formatting
    coloraxis_colorbar = dict(
    title_font_color = '#FFA028',
    tickfont_color = '#FFA028',
#    backgroundcolor = '#000000',
    bordercolor = '#333333',
    borderwidth = 1
    )
)

# 2.5. Structural frame tuning (Axes Line Components)
fig.update_xaxes(
    tickangle = -45,
    tickfont_color = '#FFA028', # Column header set to Amber
    gridcolor = '#222222', # Highly muted structural lattic gridlines
    zeroline = False
) 

fig.update_yaxes(
    scaleanchor = None, # Keeps the horizontal rectangular cell mapping active
    tickfont_color = '#FFA028', # Row labels set to Amber
    gridcolor = '#222222', # Highly muted structural lattic gridlines
    zeroline = False
) 

# 3. Display the plot (opens in browser of notebook)
fig.show()

In [ ]:
# fit the model (using 1 lag as standard for quarterly macroeconomic feedback)
results = model.fit(maxlags = 1)
print("\n--- VAR Estimation Summary ---")
print(results.summary())

In [ ]:
# Step 5: visualize interdependence (IRF)
# Traces how a 1- standard-deviation shock to one GDP affects others over 8 quarters.

irf = results.irf(periods = 8)
irf.plot(orth=True)
plt.tight_layout()
plt.show()

